# Regularización de pesos

**Capítulo 3 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_linear-regression/weight-decay.ipynb` · [Lección original](https://d2l.ai/chapter_linear-regression/weight-decay.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Decaimiento de pesos
<a id="sec_weight_decay"></a>

Ahora que hemos caracterizado el problema del exceso de ajuste, podemos introducir nuestra primera técnica de *regularización*. Recordemos que siempre podemos mitigar el exceso de ajuste recopilando más datos de entrenamiento. Sin embargo, eso puede ser costoso, consumir tiempo, o totalmente fuera de nuestro control, haciendo imposible en el corto plazo. Por ahora, podemos asumir que ya tenemos tantos datos de alta calidad como nuestros recursos lo permiten y enfocar las herramientas a nuestra disposición cuando el conjunto de datos se toma como un dado.

Recordemos que en nuestro ejemplo de regresión polinómica ([Referencia subsec_polynomial-curve-fitting](https://d2l.ai/chapter_linear-regression/generalization.html#subsec-polynomial-curve-fitting)) podríamos limitar la capacidad de nuestro modelo ajustando el grado del polinomio ajustado. De hecho, limitar el número de características es una técnica popular para mitigar el exceso de ajuste. Sin embargo, simplemente tirar a un lado las características pueden ser demasiado contundentes un instrumento. Pegando con el ejemplo de regresión polinómica, considere lo que podría suceder con la entrada de alta dimensión. Las extensiones naturales de polinomios a datos multivariados se llaman *monomios*, que son simplemente productos de poderes de variables. El grado de un monomio es la suma de las potencias. Por ejemplo, $x_1^2 x_2$, y $x_3 x_5^2$ son monomios de grado 3.

Tenga en cuenta que el número de términos con grado $d$ explota rápidamente a medida que $d$ crece. Dadas las variables $k$, el número de monomios de grado $d$ es ${k - 1 + d} \choose {k - 1}$. Incluso pequeños cambios de grado, por ejemplo, de $2$ a $3$, aumentan dramáticamente la complejidad de nuestro modelo. Por lo tanto, a menudo necesitamos una herramienta más fina para ajustar la complejidad de la función.


In [ ]:
%matplotlib inline
import torch
from torch import nn
from laboratorio import d2l

## Normas y decaimiento de pesos
** En lugar de manipular directamente el número de parámetros, *caída del peso*, opera restringiendo los valores que los parámetros pueden tomar.** Más comúnmente llamada regularización $\ell_2$ fuera de círculos de aprendizaje profundo cuando optimizada por descenso por gradiente estocástico minibatch, la decadencia del peso podría ser la técnica más utilizada para regularizar los modelos de aprendizaje automático paramétrico. La técnica está motivada por la intuición básica de que entre todas las funciones $f$, la función $f = 0$ (asignando el valor $0$ a todas las entradas) es en cierto sentido la *simple*, y que podemos medir la complejidad de una función por la distancia de sus parámetros de cero. Pero, ¿con qué precisión debemos medir la distancia entre una función y cero? No hay una única respuesta correcta. De hecho, ramas enteras de matemáticas, incluyendo partes del análisis funcional y la teoría de los espacios Banach, se dedican a abordar tales cuestiones.

Una interpretación simple podría ser medir la complejidad de una función lineal $f(\mathbf{x}) = \mathbf{w}^\top \mathbf{x}$ por alguna norma de su vector de peso, por ejemplo, $\| \mathbf{w} \|^2$. Recordemos que hemos introducido el $\ell_2$ normas y normas $\ell_1$ normas, que son casos especiales de los más $\ell_p$ norma, en [Referencia subsec_lin-algebra-norms](https://d2l.ai/chapter_preliminaries/linear-algebra.html#subsec-lin-algebra-norms). El método más común para asegurar un pequeño vector de peso es añadir su norma como un término penal al problema de minimizar la pérdida. Así reemplazamos nuestro objetivo original, * minimizando la pérdida de predicción en las etiquetas de entrenamiento*, con nuevo objetivo, * minimizando la suma de la pérdida de predicción y el término penal. Ahora, si nuestro vector de peso crece demasiado grande, nuestro algoritmo de aprendizaje podría centrarse en minimizar la norma de peso $\| \mathbf{w} \|^2$ En lugar de minimizar el error de entrenamiento. Eso es exactamente lo que queremos. Para ilustrar las cosas en código, revivimos nuestro ejemplo anterior de [Referencia sec_linear_regression](https://d2l.ai/chapter_linear-regression/linear-regression.html#sec-linear-regression) para la regresión lineal. Allí, nuestra pérdida fue dada por

$$L(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^n \frac{1}{2}\left(\mathbf{w}^\top \mathbf{x}^{(i)} + b - y^{(i)}\right)^2.$$

Recordemos que $\mathbf{x}^{(i)}$ son las características, $y^{(i)}$ es la etiqueta para cualquier dato ejemplo $i$, y $(\mathbf{w}, b)$ son los parámetros de peso y sesgo, respectivamente. Para penalizar el tamaño del vector de peso, de alguna manera debemos añadir $\| \mathbf{w} \|^2$ a la función de pérdida, pero ¿cómo debe el modelo cambiar la pérdida estándar para este nuevo aditivo penalización? En la práctica, caracterizamos esta compensación a través de la *regularización constante* $\lambda$, un hiperparametro no negativo que encajamos con datos de validación:

$$L(\mathbf{w}, b) + \frac{\lambda}{2} \|\mathbf{w}\|^2.$$

Para $\lambda = 0$, recuperamos nuestra función de pérdida original. Para $\lambda > 0$, restringimos el tamaño de $\| \mathbf{w} \|$. Dividimos por $2$ por convención: cuando tomamos la derivada de una función cuadrática, las $2$ y $1/2$ cancelan, asegurando que la expresión para la actualización se vea agradable y simple. El lector astuto podría preguntarse por qué trabajamos con la norma cuadrada y no la norma estándar (es decir, la distancia euclidiana). Lo hacemos para comodidad computacional. Al cuadrar la norma $\ell_2$, eliminamos la raíz cuadrada, dejando la suma de cuadrados de cada componente del vector de peso. Esto hace que la derivada de la pena sea fácil de calcular: la suma de derivados es igual a la derivada de la suma.

Por otra parte, se puede preguntar por qué trabajamos con la norma $\ell_2$ en primer lugar y no, digamos, la norma $\ell_1$. De hecho, otras opciones son válidas y populares en todas las estadísticas. Mientras que los modelos lineales regulados $\ell_2$ constituyen el algoritmo clásico *regresión del ridge*, la regresión lineal regularizada $\ell_1$ es un método igualmente fundamental en las estadísticas, popularmente conocido como *regresión de láser*. Una razón para trabajar con la norma $\ell_2$ es que coloca una penalización desmesurada en grandes componentes del vector de peso. Este sesgo nuestro algoritmo de aprendizaje hacia modelos que distribuyen el peso uniformemente a través de un mayor número de características. En la práctica, esto podría hacerlos más robustos al error de medición en una sola variable. Por el contrario, las penalizaciones $\ell_1$ conducen a modelos que concentran pesos en un pequeño conjunto de características al eliminar los otros pesos a cero. Esto nos da un método eficaz para la selección de funciones*, que puede ser deseable por otras razones.

Usando la misma notación en [Referencia eq_linreg_batch_update](https://d2l.ai/#eq-linreg-batch-update), las actualizaciones de descenso por gradiente estocástico minibatch para regresión regularizada $\ell_2$ de la siguiente manera:

$$\begin{aligned}
\mathbf{w} & \leftarrow \left(1- \eta\lambda \right) \mathbf{w} - \frac{\eta}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \mathbf{x}^{(i)} \left(\mathbf{w}^\top \mathbf{x}^{(i)} + b - y^{(i)}\right).
\end{aligned}$$

Como antes, actualizamos $\mathbf{w}$ en base a la cantidad por la cual nuestra estimación difiere de la observación. Sin embargo, también reducimos el tamaño de $\mathbf{w}$ hacia cero. Es por eso que el método a veces se llama "caída de peso": dado el término penal, nuestro algoritmo de optimización *decae* el peso en cada paso de entrenamiento. En contraste con la selección de características, la descomposición de peso nos ofrece un mecanismo para ajustar continuamente la complejidad de una función. Valores más pequeños de $\lambda$ corresponden a $\mathbf{w}$ menos limitado, mientras que valores más grandes de $\lambda$ limitan $\mathbf{w}$ con mássiderablemente. Si incluimos una penalización de sesgo correspondiente $b^2$ puede variar entre implementaciones, y puede variar entre capas de una red neuronal. A menudo, no regularizamos el término de sesgo. Además, aunque $\ell_2$ la regularización no puede ser equivalente a la descomposición de peso para otros algoritmos de optimización, la idea de regularización mediante la reducción del tamaño de los pesos sigue siendo cierta.

## Regresión lineal de alta dimensión
Podemos ilustrar los beneficios de la decaimiento de pesos a través de un simple ejemplo sintético.

Primero, **generamos algunos datos como antes**:

**

$$y = 0.05 + \sum_{i = 1}^d 0.01 x_i + \epsilon \textrm{ where }
\epsilon \sim \mathcal{N}(0, 0.01^2).$$

**

En este conjunto de datos sintéticos, nuestra etiqueta está dada por una función lineal subyacente de nuestras entradas, corrompida por el ruido gaussiano con media cero y desviación estándar 0.01. Para fines ilustrativos, podemos hacer que los efectos de sobreajuste pronunciado, aumentando la dimensionalidad de nuestro problema a $d = 200$ y trabajando con un pequeño conjunto de entrenamiento con sólo 20 ejemplos.


In [ ]:
class Data(d2l.DataModule):
    def __init__(self, num_train, num_val, num_inputs, batch_size):
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, num_inputs)
        noise = torch.randn(n, 1) * 0.01
        w, b = torch.ones((num_inputs, 1)) * 0.01, 0.05
        self.y = torch.matmul(self.X, w) + b + noise

    def get_dataloader(self, train):
        i = slice(0, self.num_train) if train else slice(self.num_train, None)
        return self.get_tensorloader([self.X, self.y], train, i)

## Implementación desde cero
Dado que el descenso por gradiente estocástico minibatch es nuestro optimizador, sólo tenemos que añadir la penalización $\ell_2$ al cuadrado a la función de pérdida original.

### Definición de la penalización de norma $\ell_2$

Tal vez la forma más conveniente de aplicar esta pena es cuadrar todos los términos en su lugar y sumarlos.


In [ ]:
def l2_penalty(w):
    return (w ** 2).sum() / 2

### Definir el modelo
En el modelo final, la regresión lineal y la pérdida cuadrada no han cambiado desde [Referencia sec_linear_scratch](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch), por lo que vamos a definir una subclase de `d2l.LinearRegressionScratch`. El único cambio aquí es que nuestra pérdida ahora incluye el término penalti.


In [ ]:
class WeightDecayScratch(d2l.LinearRegressionScratch):
    def __init__(self, num_inputs, lambd, lr, sigma=0.01):
        super().__init__(num_inputs, lr, sigma)
        self.save_hyperparameters()

    def loss(self, y_hat, y):
        return (super().loss(y_hat, y) +
                self.lambd * l2_penalty(self.w))

El siguiente código se ajusta a nuestro modelo en el conjunto de entrenamiento con 20 ejemplos y lo evalúa en el conjunto de validación con 100 ejemplos.


In [ ]:
data = Data(num_train=20, num_val=100, num_inputs=200, batch_size=5)
trainer = d2l.Trainer(max_epochs=10)

def train_scratch(lambd):
    model = WeightDecayScratch(num_inputs=200, lambd=lambd, lr=0.01)
    model.board.yscale='log'
    trainer.fit(model, data)
    print('L2 norm of w:', float(l2_penalty(model.w)))

### Nota docente de Hespérides

Para comparar modelos, conserva la partición y empareja las semillas. Selecciona hiperparámetros con validación y reserva el test para el final. La versión D2L de Fashion-MNIST llama «val» al test oficial: en estos derivados se separa validación del entrenamiento oficial. El modo rápido demuestra mecanismos; no permite extraer una clasificación definitiva de técnicas.

Vínculo con los apuntes: sesión 3, «Regularización de pesos».


### Entrenamiento sin regularización

Ahora ejecutamos este código con `lambd = 0`, deshabilitando el deterioro del peso. Tenga en cuenta que nos sobreencajamos mal, disminuyendo el error de entrenamiento pero no el error de validación--un caso de libro de texto de exceso de ajuste.


In [ ]:
train_scratch(0)

### Uso de la decaimiento de pesos

A continuación, corremos con decaimiento de peso sustancial. Tenga en cuenta que el error de entrenamiento aumenta pero el error de validación disminuye. Este es precisamente el efecto que esperamos de la regularización.


In [ ]:
train_scratch(3)

## Implementación concisa

Debido a que el deterioro del peso es omnipresente en la optimización de la red neuronal, el biblioteca de aprendizaje profundo lo hace especialmente conveniente, integrando el deterioro del peso en el propio algoritmo de optimización para su fácil uso en combinación con cualquier función de pérdida. Además, esta integración sirve para un beneficio computacional, permitiendo que los trucos de implementación añadan deterioro del peso al algoritmo, sin ningún exceso computacional adicional.


A continuación, especificamos el hiperparametro de deterioro de peso directamente a través de `weight_decay` al instanciar nuestro optimizador. Por defecto, PyTorch decae simultáneamente tanto pesos como sesgos, pero podemos configurar el optimizador para manejar diferentes parámetros de acuerdo con diferentes políticas. Aquí, sólo establecemos `weight_decay` para los pesos (los parámetros `net.weight`), por lo que el sesgo (el parámetro `net.bias`) no se decaerá.


In [ ]:
class WeightDecay(d2l.LinearRegression):
    def __init__(self, wd, lr):
        super().__init__(lr)
        self.save_hyperparameters()
        self.wd = wd

    def configure_optimizers(self):
        return torch.optim.SGD([
            {'params': self.net.weight, 'weight_decay': self.wd},
            {'params': self.net.bias}], lr=self.lr)

**La trama se ve similar a la que implementamos cuando el peso se decae desde cero**. Sin embargo, esta versión se ejecuta más rápido y es más fácil de implementar, beneficios que se harán más pronunciados a medida que se abordan problemas más grandes y este trabajo se vuelve más rutinario.


In [ ]:
model = WeightDecay(wd=3, lr=0.01)
model.board.yscale='log'
trainer.fit(model, data)

print('L2 norm of w:', float(l2_penalty(model.get_w_b()[0])))

Hasta ahora, hemos tocado una noción de lo que constituye una función lineal simple. Sin embargo, incluso para funciones no lineales simples, la situación puede ser mucho más compleja. Para ver esto, el concepto de [reproducing kernel Hilbert space (RKHS)](https://en.wikipedia.org/wiki/Reproducing_kernel_Hilbert_space) permite aplicar herramientas introducidas para funciones lineales en un contexto no lineal. Desafortunadamente, los algoritmos basados en RKHS tienden a escalar pobremente a datos grandes, de alta dimensión. En este libro a menudo adoptaremos la heurística común por la que la descomposición de peso se aplica a todas las capas de una red profunda.

## Resumen
Regularización es un método común para lidiar con el exceso de ajuste. Técnicas clásicas de regularización añaden un término penal a la función de pérdida (cuando el entrenamiento) para reducir la complejidad del modelo aprendido. Una opción particular para mantener el modelo simple es utilizar una penalización $\ell_2$. Esto conduce a la descomposición de peso en los pasos de actualización del algoritmo de descenso por gradiente estocástico minibatch. En la práctica, la funcionalidad de deterioro de peso se proporciona en optimizadores de bibliotecas de aprendizaje profundo. Diferentes conjuntos de parámetros pueden tener diferentes comportamientos de actualización dentro del mismo bucle de entrenamiento.

## Ejercicios
1. Experimenta con el valor de $\lambda$ en el problema de estimación en esta sección. Entrenamiento de gráficos y precisión de validación en función de $\lambda$. ¿Qué observas?
1. Utilice un conjunto de validación para encontrar el valor óptimo de $\lambda$. ¿Es realmente el valor óptimo? ¿Importa esto?
1. ¿Cómo serían las ecuaciones de actualización si en lugar de $\|\mathbf{w}\|^2$ usamos $\sum_i |w_i|$ como nuestra penalización de elección (regularización $\ell_1$)?
1. Sabemos que $\|\mathbf{w}\|^2 = \mathbf{w}^\top \mathbf{w}$. ¿Se puede encontrar una ecuación similar para matrices (ver la norma Frobenius en [Referencia subsec_lin-algebra-norms](https://d2l.ai/chapter_preliminaries/linear-algebra.html#subsec-lin-algebra-norms))?
1. Revisar la relación entre error de entrenamiento y error de generalización. Además de la decaimiento de pesos, el aumento del entrenamiento y el uso de un modelo de complejidad adecuada, ¿qué otras maneras podrían ayudarnos a lidiar con el exceso de ajuste?
1. En las estadísticas bayesianas utilizamos el producto de anterior y la probabilidad de llegar a una posterior vía $P(w \mid x) \propto P(x \mid w) P(w)$. ¿Cómo se puede identificar $P(w)$ con la regularización?


[Debate del original](https://discuss.d2l.ai/t/99)
